# **Read CSV**

In [5]:
import pandas as pd

In [6]:
order_items_df = pd.read_csv("Data/order_items.csv")

In [8]:
order_df = pd.read_csv("Data/orders.csv")

In [9]:
payments_df = pd.read_csv("Data/orders.csv")

In [10]:
products_df = pd.read_csv("Data/products.csv")

In [11]:
users_df = pd.read_csv("Data/users.csv")

---

In [8]:
import sqlalchemy as sal

In [7]:
import pandas as pd
from sqlalchemy import create_engine, text
from tqdm import tqdm

In [9]:
# ==========================
# CONFIG
# ==========================
DB_USER = "shivam"
DB_PASSWORD = "2005"
DB_HOST = "localhost"
DB_PORT = "3306"
DB_NAME = "ecommerce_db"

CHUNK_SIZE = 5000

# ==========================
# CREATE ENGINE
# ==========================
engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}"
)

# ==========================
# CREATE DATABASE
# ==========================

In [25]:
with engine.connect() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}"))
    print("Database ensured")

Database ensured


In [ ]:
# reconnect to specific DB
engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# ==========================
# CREATE TABLES
# ==========================
schema_sql = """
CREATE TABLE IF NOT EXISTS users (
    user_id VARCHAR(36) PRIMARY KEY,
    name TEXT,
    email TEXT,
    phone TEXT,
    city TEXT,
    country TEXT,
    created_at DATETIME
);

CREATE TABLE IF NOT EXISTS products (
    product_id VARCHAR(36) PRIMARY KEY,
    name TEXT,
    category TEXT,
    price FLOAT,
    stock INT
);

CREATE TABLE IF NOT EXISTS orders (
    order_id VARCHAR(36) PRIMARY KEY,
    user_id VARCHAR(36),
    order_date DATETIME,
    status TEXT,
    FOREIGN KEY (user_id) REFERENCES users(user_id)
);

CREATE TABLE IF NOT EXISTS order_items (
    order_item_id VARCHAR(36) PRIMARY KEY,
    order_id VARCHAR(36),
    product_id VARCHAR(36),
    quantity INT,
    FOREIGN KEY (order_id) REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);

CREATE TABLE IF NOT EXISTS payments (
    payment_id VARCHAR(36) PRIMARY KEY,
    order_id VARCHAR(36),
    amount FLOAT,
    payment_method TEXT,
    payment_status TEXT,
    payment_date DATETIME,
    FOREIGN KEY (order_id) REFERENCES orders(order_id)
);
"""

with engine.connect() as conn:
    for stmt in schema_sql.split(";"):
        if stmt.strip():
            conn.execute(text(stmt))
    print("✅ Tables created")

# ==========================
# LOAD CSV IN CHUNKS
# ==========================
def load_csv_to_mysql(file_path, table_name):
    print(f"🚀 Loading {table_name}...")

    for chunk in tqdm(pd.read_csv(file_path, chunksize=CHUNK_SIZE)):
        chunk.to_sql(
            table_name,
            con=engine,
            if_exists="append",
            index=False,
            method="multi"  # faster insert
        )

    print(f"✅ Finished {table_name}")

# ==========================
# INGEST DATA
# ==========================
def main():
    load_csv_to_mysql("Data/users.csv", "users")
    load_csv_to_mysql("Data/products.csv", "products")
    load_csv_to_mysql("Data/orders.csv", "orders")
    load_csv_to_mysql("Data/order_items.csv", "order_items")
    load_csv_to_mysql("Data/payments.csv", "payments")

    print("🎉 All data loaded successfully!")

if __name__ == "__main__":
    main()

✅ Tables created
🚀 Loading users...


0it [00:00, ?it/s]

20it [00:12,  1.65it/s]


✅ Finished users
🚀 Loading products...


4it [00:01,  3.25it/s]


...

✅ Finished orders
🚀 Loading order_items...


180it [05:44,  1.91s/it]


✅ Finished order_items
🚀 Loading payments...


60it [00:35,  1.68it/s]

✅ Finished payments
🎉 All data loaded successfully!


In [10]:
query = """
    SELECT * FROM users limit 10;
"""

In [11]:
engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

In [15]:
with engine.connect() as conn:
    result = pd.DataFrame(conn.execute(text(query)))

In [17]:
result

,user_id,name,email,phone,city,country,created_at
0,0000d817-629f-436b-b1e7-b71fe1102499,Misty Jackson,maldonadojacob@example.org,001-824-487-1933x09066,Victoriastad,Qatar,2023-09-18 10:21:35
1,000196c5-99a4-4c4a-96ef-3ba48a564b85,Michael Harris,georgeburton@example.net,(856)557-5675,Reynoldsburgh,Estonia,2020-03-26 13:21:42
2,00019ae4-fbf6-4589-840d-c142cbdcf552,David Durham,leejames@example.net,608.528.1438x097,Kingville,Antigua and Barbuda,2020-02-09 06:10:11
3,00021d12-efcb-41a4-86b0-dd4b08f359d8,Nicole Morales,hardyalex@example.com,001-325-376-7108,Levybury,Italy,2024-08-27 04:10:38
4,0003fccd-3076-41ef-beb4-28ecebcf9971,Jordan James,katrinalang@example.org,001-386-402-4096,East Karenbury,Saudi Arabia,2023-12-04 11:01:53
5,000600eb-cc78-433c-a7d0-4347c81478cd,Chase Lyons,cluna@example.com,+1-256-867-5242x364,East Evan,Norway,2024-10-01 23:13:55
6,000614d8-ab87-48a4-825b-a319b53c3abe,Andrew Hurst,kimberly42@example.com,936.550.7954,East Ronald,Belize,2022-03-29 11:19:55
7,000669b2-a8d5-45e7-8a49-313cd6a3352f,David Petersen,joeschwartz@example.org,+1-869-646-3026x6720,Gilbertshire,Yemen,2024-08-14 16:25:07
8,000685e8-a046-42b4-bda2-99faf6f2dfb8,Cindy Ward,franciscolopez@example.net,(373)897-4198,Rossport,Congo,2021-08-16 00:35:58
9,0006bba7-2638-4a9e-94cc-46404f4dc955,Timothy Walker,choipaul@example.com,874-261-5713x35520,East Julieview,Vanuatu,2025-11-19 19:34:40


### Indexiing

In [7]:
query = """
CREATE INDEX idx_orders_user_id ON orders(user_id);
CREATE INDEX idx_order_items_order_id ON order_items(order_id);
CREATE INDEX idx_order_items_product_id ON order_items(product_id);
CREATE INDEX idx_payments_order_id ON payments(order_id);


CREATE INDEX idx_products_price ON products(price);
CREATE INDEX idx_orders_date ON orders(order_date);
CREATE INDEX idx_payments_status ON payments(payment_status);"""

In [8]:
import sqlalchemy as sal

In [9]:
DB_USER = "shivam"
DB_PASSWORD = "2005"
DB_HOST = "localhost"
DB_PORT = "3306"
DB_NAME = "ecommerce_db"

engine = sal.create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}"
)

In [16]:
from sqlalchemy import text

indexes = [
    "CREATE INDEX idx_orders_user_id ON orders(user_id)",
    "CREATE INDEX idx_order_items_order_id ON order_items(order_id)",
    "CREATE INDEX idx_order_items_product_id ON order_items(product_id)",
    "CREATE INDEX idx_payments_order_id ON payments(order_id)",
    "CREATE INDEX idx_products_price ON products(price)",
    "CREATE INDEX idx_orders_date ON orders(order_date)",]

In [18]:
import pandas as pd

In [20]:
QUERY = """SELECT 
    TABLE_NAME,
    INDEX_NAME,
    COLUMN_NAME,
    NON_UNIQUE
FROM information_schema.STATISTICS
WHERE TABLE_SCHEMA = 'ecommerce_db'
ORDER BY TABLE_NAME, INDEX_NAME;"""

with engine.connect() as conn:
    conn.execute(text("use ecommerce_db;"))
    RESULT = conn.execute(text(QUERY))
    print(pd.DataFrame(RESULT))

     TABLE_NAME                  INDEX_NAME    COLUMN_NAME  NON_UNIQUE
0   order_items    idx_order_items_order_id       order_id           1
1   order_items  idx_order_items_product_id     product_id           1
2   order_items                     PRIMARY  order_item_id           0
3        orders             idx_orders_date     order_date           1
4        orders          idx_orders_user_id        user_id           1
5        orders                     PRIMARY       order_id           0
6      payments       idx_payments_order_id       order_id           1
7      payments                     PRIMARY     payment_id           0
8      products          idx_products_price          price           1
9      products                     PRIMARY     product_id           0
10        users                     PRIMARY        user_id           0
